# Benford 定律的卡方检验

> 对应代码：`MathStatsCode/code_in_notes/Chap.14/Benford.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：检验2011年城市GDP首位数字是否服从 Benford 定律（χ²(8) 检验）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.14`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.14")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all` 清空内存，载入城市数据并保留2011年的样本（`keep if Year==2011`）。本 notebook 检验 **Benford 定律**：自然产生的数据（如GDP）首位数字为 d 的概率约为 log10(1+1/d)——1出现约30%、9只出现约4.6%。


In [ ]:
%%stata
clear all
use ../datasets/citydata.dta
keep if Year==2011


**将GDP转为字符串**

`tostring v84, gen(gdp_string)`：把数值型 GDP 变量转换为字符串（存为 gdp_string），这样就能按“字符位置”提取首位数字；`tostring` 的 `gen()` 选项保留原变量。


In [ ]:
%%stata
tostring v84, gen(gdp_string)


**取出GDP数字字符串的第一位**

`gen first_digit=substr(gdp_string,1,1)`：`substr(s,1,1)` 截取字符串的第1个字符——即 GDP 的**首位数字**（仍以字符形式存储）。


In [ ]:
%%stata
gen first_digit=substr(gdp_string, 1,1)


**将第一位数字字符串转化为数字**

`destring first_digit, replace`：把字符型的首位数字转换回数值型（覆盖原变量），以便统计频数并进行计算。


In [ ]:
%%stata
destring first_digit, replace


**统计频数**

`tab first_digit`：输出首位数字的频数分布。把它与 Benford 定律的理论概率对比（1最多、数字越大越少），可以先用眼睛做个初步判断。


In [ ]:
%%stata
tab first_digit


**创建一个数据框用于检验**

这一段用**卡方拟合优度检验**完成正式检验（H0：首位数字服从 Benford 分布）：
- `frame create chi2test Ng Ng_star` 新建结果数据框，用于存放每个数字的**实际频数** Ng 与**理论频数** Ng_star；
- 先 `su first_digit` 取得总样本数 N，然后 `forvalues i=1/9` 对数字 1~9 循环：实际频数由 `su ... if first_digit==i` 得到，理论频数按 Benford 概率计算 `Ng_star=N×[log10(i+1)-log10(i)]`，用 `frame post` 写入；
- 在新框中计算每个数字的卡方贡献 `(Ng-Ng_star)²/Ng_star`，求和得到检验统计量 `test_stat`；
- 自由度为 9-1=8，用 `1-chi2(8, test_stat)` 计算 p 值（`chi2(df,x)` 是 χ² 分布的累积概率）。p 值大到一定程度即不能拒绝“GDP首位数字服从 Benford 定律”。


In [ ]:
%%stata
frame create chi2test Ng Ng_star
quietly: su first_digit
local N=r(N)
forvalues i=1/9{
	quietly: su first_digit if first_digit==`i'
	local Ng=r(N)
	local Ng_star=`N'*(log10(`i'+1)-log10(`i'))
	frame post chi2test (`Ng') (`Ng_star')
}
frame chi2test: gen test_stat=(Ng-Ng_star)^2/Ng_star
frame chi2test: su test_stat
frame chi2test: local test_stat=r(sum)
di "test statistics= `test_stat'"
local p=1-chi2(8 ,`test_stat')
di "p-value= `p'" 
